# MH-03. X (baseline <= index): auto-wide raw + mask + days_since

Goal: take as MANY features as possible (wide), leave the 5% coverage gate and reduction to MH-04.

- Numeric blocks (measurement, observation): auto-pull EVERY concept whose coverage over the
  cohort >= MIN_COVERAGE_WIDE (a loose pre-gate, looser than MH-04's real 5% gate). For each,
  raw_value = latest value on-or-before index (ties by last source id, deterministic),
  observed_mask = 1 if any value <= index, days_since = index - obs date.
- Drug block: binary pre-index exposure per drug concept (0/1, fully observed, no mask/days_since).
- Observation-intensity block: visit counts, kept raw.

Heavy as-of-index picking is done in DuckDB (window functions), not pandas. `days_since` is
delivered for audit but excluded from the reducer in MH-04.


## 1. Config

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import duckdb

DATA_PATH = '/home/jupyter/2836994-data2'
OUTPUT_DIR = Path('output/mh')
con = duckdb.connect()

spine = pd.read_parquet(OUTPUT_DIR / 'spine.parquet')[['person_id','index_date']]
spine['index_date'] = pd.to_datetime(spine['index_date'])
con.register('spine', spine)
persons = sorted(spine['person_id'].tolist())
N_PERSONS = len(persons)

# Wide pre-gate: keep any concept observed (<= index) in at least this fraction of the cohort.
# Looser than MH-04's real 5% gate on purpose (wide now, gate later).
MIN_COVERAGE_WIDE = 0.01
MIN_PERSONS = int(np.ceil(MIN_COVERAGE_WIDE * N_PERSONS))
# Safety cap per block (take highest-coverage concepts first) to bound width/runtime.
MAX_FEATURES_PER_BLOCK = 3000
INCLUDE_DRUG_BINARY = True

def gp(table):
    return f"read_parquet('{DATA_PATH}/{table}/*.parquet')"

print(f'cohort persons: {N_PERSONS} | wide pre-gate: >= {MIN_PERSONS} persons ({MIN_COVERAGE_WIDE:.0%})')


## 2. Generic numeric as-of-index builder (DuckDB window functions)

In [ ]:
def build_numeric_wide(table, concept_col, value_col, date_col, id_col, block):
    """Auto-pull every numeric concept with coverage >= MIN_PERSONS; return wide
    raw / mask / days_since reindexed to the full cohort, columns keyed by concept_id."""
    date_expr = f'CAST({date_col} AS DATE)'
    # 1) coverage-passing concepts (observed on-or-before index).
    sel = con.sql(f'''
        SELECT t.{concept_col} AS cid, COUNT(DISTINCT t.person_id) AS n_persons
        FROM {gp(table)} t JOIN spine s ON t.person_id = s.person_id
        WHERE t.{value_col} IS NOT NULL AND {date_expr} <= s.index_date
              AND t.{concept_col} IS NOT NULL AND t.{concept_col} <> 0
        GROUP BY 1 HAVING COUNT(DISTINCT t.person_id) >= {MIN_PERSONS}
        ORDER BY n_persons DESC LIMIT {MAX_FEATURES_PER_BLOCK}
    ''').to_df()
    if len(sel) == 0:
        print(f'[{block}] no concept passed the wide pre-gate'); return None, sel
    con.register(f'sel_{block}', sel[['cid']])

    # 2) latest value on-or-before index per (person, concept), deterministic tie-break.
    picked = con.sql(f'''
        WITH ev AS (
          SELECT t.person_id, t.{concept_col} AS cid,
                 {date_expr} AS d, t.{value_col} AS val, t.{id_col} AS sid, s.index_date
          FROM {gp(table)} t JOIN spine s ON t.person_id = s.person_id
          JOIN sel_{block} ON sel_{block}.cid = t.{concept_col}
          WHERE t.{value_col} IS NOT NULL AND {date_expr} <= s.index_date
        ), ranked AS (
          SELECT *, row_number() OVER (PARTITION BY person_id, cid ORDER BY d DESC, sid DESC) rn
          FROM ev
        )
        SELECT person_id, cid, val AS raw_value,
               date_diff('day', d, index_date) AS days_since
        FROM ranked WHERE rn = 1
    ''').to_df()

    raw = picked.pivot(index='person_id', columns='cid', values='raw_value').reindex(persons)
    dsi = picked.pivot(index='person_id', columns='cid', values='days_since').reindex(persons)
    mask = raw.notna().astype('int8')
    raw.columns  = [f'{block}__{c}__raw' for c in raw.columns]
    mask.columns = [f'{block}__{c}__mask' for c in mask.columns]
    dsi.columns  = [f'{block}__{c}__days_since' for c in dsi.columns]
    wide = pd.concat([raw, mask, dsi], axis=1)
    print(f'[{block}] concepts kept: {len(sel)} -> columns: {wide.shape[1]}')
    return wide, sel


## 3. Measurement + Observation numeric blocks (auto-wide)

In [ ]:
blocks, manifests = [], []

meas_wide, meas_sel = build_numeric_wide(
    'measurement', 'measurement_concept_id', 'value_as_number',
    'measurement_date', 'measurement_id', 'Meas')
if meas_wide is not None:
    blocks.append(meas_wide); meas_sel['block'] = 'Meas'; manifests.append(meas_sel)

obs_wide, obs_sel = build_numeric_wide(
    'observation', 'observation_concept_id', 'value_as_number',
    'observation_date', 'observation_id', 'Obs')
if obs_wide is not None:
    blocks.append(obs_wide); obs_sel['block'] = 'Obs'; manifests.append(obs_sel)


## 4. Drug block: binary pre-index exposure (0/1, fully observed)

In [ ]:
if INCLUDE_DRUG_BINARY:
    drug_sel = con.sql(f'''
        SELECT d.drug_concept_id AS cid, COUNT(DISTINCT d.person_id) AS n_persons
        FROM {gp('drug_exposure')} d JOIN spine s ON d.person_id = s.person_id
        WHERE CAST(d.drug_exposure_start_date AS DATE) <= s.index_date
              AND d.drug_concept_id IS NOT NULL AND d.drug_concept_id <> 0
        GROUP BY 1 HAVING COUNT(DISTINCT d.person_id) >= {MIN_PERSONS}
        ORDER BY n_persons DESC LIMIT {MAX_FEATURES_PER_BLOCK}
    ''').to_df()
    if len(drug_sel):
        con.register('drug_sel', drug_sel[['cid']])
        pres = con.sql(f'''
            SELECT DISTINCT d.person_id, d.drug_concept_id AS cid
            FROM {gp('drug_exposure')} d JOIN spine s ON d.person_id = s.person_id
            JOIN drug_sel ON drug_sel.cid = d.drug_concept_id
            WHERE CAST(d.drug_exposure_start_date AS DATE) <= s.index_date
        ''').to_df()
        pres['v'] = np.int8(1)
        drug_wide = (pres.pivot(index='person_id', columns='cid', values='v')
                     .reindex(persons).fillna(0).astype('int8'))
        drug_wide.columns = [f'Drug__{c}__raw' for c in drug_wide.columns]
        blocks.append(drug_wide); drug_sel['block'] = 'Drug'; manifests.append(drug_sel)
        print(f'[Drug] concepts kept: {len(drug_sel)} -> columns: {drug_wide.shape[1]}')
    else:
        print('[Drug] no concept passed the wide pre-gate')


## 5. Observation-intensity block (counts, kept raw)

In [ ]:
obs_int = con.sql(f'''
WITH v AS (SELECT person_id, CAST(visit_start_date AS DATE) AS d FROM {gp('visit_occurrence')})
SELECT s.person_id,
       COUNT(v.d)          AS prior_encounters,
       COUNT(DISTINCT v.d) AS distinct_visit_days
FROM spine s LEFT JOIN v ON v.person_id = s.person_id AND v.d <= s.index_date
GROUP BY s.person_id
''').to_df().set_index('person_id').reindex(persons)
obs_int.columns = [f'ObsIntensity__{c}__raw' for c in obs_int.columns]
blocks.append(obs_int.fillna(0))
print('[ObsIntensity] columns:', obs_int.shape[1])


## 6. Assemble X + feature manifest (row-aligned, ascending by person_id)

In [ ]:
X = pd.concat(blocks, axis=1) if blocks else pd.DataFrame(index=persons)
X.index.name = 'person_id'
X = X.sort_index().reset_index()
assert X['person_id'].tolist() == sorted(persons)
X.to_parquet(OUTPUT_DIR / 'X_raw.parquet', index=False)

# Feature manifest: which concept each column came from, with coverage + concept_name.
if manifests:
    man = pd.concat(manifests, ignore_index=True)
    names = con.sql(f'''
        SELECT concept_id AS cid, concept_name FROM {gp('concept')}
    ''').to_df()
    man = man.merge(names, on='cid', how='left')
    man['coverage'] = man['n_persons'] / N_PERSONS
    man = man.sort_values(['block','n_persons'], ascending=[True, False])
    man.to_csv(OUTPUT_DIR / 'X_feature_manifest.csv', index=False)
    print('feature manifest rows:', len(man))

print('X_raw:', X.shape)
print('columns by suffix:',
      {s: sum(c.endswith(s) for c in X.columns) for s in ['__raw','__mask','__days_since']})
print('columns by block:',
      {b: sum(c.startswith(b + '__') for c in X.columns)
       for b in ['Meas','Obs','Drug','ObsIntensity']})
